### Import Dependencies

In [ ]:
import openai
import pandas as pd

from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
    Prefetch,
    Fusion,
    FusionQuery,
)

### Create Qdrant collection for hybrid search

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE),
        },
        sparse_vectors_config={
            "bm25": SparseVectorParams(
                modifier=Modifier.IDF,
            )
        },
    )


In [ ]:
qdrant_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD,
)

### Embedding Functions

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


In [ ]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
  if len(text_list) <= batch_size:
    response = openai.embeddings.create(
      input=text_list,
      model=model
    )
    return [item.embedding for item in response.data]
  
  all_embeddings = []
  counter = 1
  for i in range(0, len(text_list), batch_size):
    batch = text_list[i:i+batch_size]
    response = openai.embeddings.create(
      input=batch,
      model=model
    )
    all_embeddings.extend([item.embedding for item in response.data])
    print(f"Processed {counter * batch_size} of {len(text_list)}")
    counter += 1

  return all_embeddings

### Read the sampled dataset with Amazon inventory

In [ ]:
sampled_items = pd.read_json(
    "../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    lines=True,
)

In [ ]:
sampled_items.head()

In [ ]:
list(sampled_items["features"].items())[0]

In [ ]:
list(sampled_items["images"].items())[0]

### Preprocess title and features

In [ ]:
def preprocess_description(row):
  return f"{row['title']} {' '.join(row['features'])}"

In [ ]:
def extract_first_large_image(row):
  return row["images"][0].get("large", "")

In [ ]:
sampled_items["preprocessed_description"] = sampled_items.apply(preprocess_description, axis=1)
sampled_items["image"] = sampled_items.apply(extract_first_large_image, axis=1)

In [ ]:
df_data_to_embed = sampled_items[["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]]


In [ ]:
df_data_to_embed.head()

In [ ]:
if (not isinstance(df_data_to_embed, pd.DataFrame)):
    raise TypeError("expected a DataFrame")
data_to_embed = df_data_to_embed.to_dict(orient="records")

In [ ]:
data_to_embed

In [ ]:
len(data_to_embed)

In [ ]:
text_to_embed = [item["preprocessed_description"] for item in data_to_embed]

In [ ]:
text_to_embed

In [ ]:
embeddings = get_embeddings_batch(text_to_embed)

In [ ]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
  pointstructs.append(
    PointStruct(
      id=i,
      vector={
        "text-embedding-3-small": embedding,
        "bm25": Document(
          text=data["preprocessed_description"],
          model="qdrant/bm25",
        ),
      },
      payload=data
    )
    
  )
  i += 1


In [ ]:
pointstructs[0].vector

In [ ]:
qdrant_client.upsert(
  collection_name=COLLECTION_NAME,
  points=pointstructs[0:500],
  wait=True
)

In [ ]:
qdrant_client.upsert(
  collection_name=COLLECTION_NAME,
  points=pointstructs[500:],
  wait=True
)

### Hybrid Retrieval

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20,
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20,
            ),
        ],
        query=FusionQuery(
            fusion=Fusion.RRF
        ),
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint", result)
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }

In [ ]:
results = retrieve_data("can I get a tablet?", k=20)

In [ ]:
results

In [ ]:
# print("\n".join([description for description in results['retrieved_context']]))

### Hybrid Search with weighted RRF

In [ ]:
def retrieve_data_biased_toward_semantic_similarity(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20,
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20,
            ),
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3, 1])),
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint", result)
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


In [ ]:
results_biased_toward_semantic_similarity = retrieve_data_biased_toward_semantic_similarity("can I get a tablet?", k=20)

In [ ]:
def print_results_comparison(results, results_biased_toward_semantic_similarity):
    def show_top_5_titles(res, label):
        print(f"\nTop 5 results for {label}:")
        for i, title in enumerate(res["retrieved_context"][:5]):
            print(f"{i+1}. (ASIN: {res['retrieved_context_ids'][i]}) Rating: {res['retrieved_context_ratings'][i]} | {title[:100]}...")

    print("="*40)
    print("Comparison of Results (default) vs Biased Toward Semantic Similarity")
    print("="*40)

    show_top_5_titles(results, "default results")
    show_top_5_titles(results_biased_toward_semantic_similarity, "biased toward semantic similarity")

    # Optionally, you could print ids of top k for side-by-side overlap
    print("\nOverlap in top 5 ASINs:")
    default_ids = set(results["retrieved_context_ids"][:5])
    biased_ids = set(results_biased_toward_semantic_similarity["retrieved_context_ids"][:5])
    print("Common ASINs:", default_ids & biased_ids)

# Example call:
print_results_comparison(results, results_biased_toward_semantic_similarity)